<a href="https://colab.research.google.com/github/faisalepty/Sign-Language-CNN/blob/main/transfer_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms, datasets
from torch.utils.data import ConcatDataset, DataLoader, random_split
import torchvision.models as models

# 1. Load pretrained model
model = models.efficientnet_b0(weights='IMAGENET1K_V1')

# 2. Freeze all layers (don't touch pretrained weights)
for param in model.parameters():
    param.requires_grad = False

# 3. Replace ONLY the final classifier for your 29 classes
model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(1280, 29)
)

# 4. Only the classifier trains, everything else is frozen
optimizer = torch.optim.AdamW(
    model.classifier.parameters(),  # only new layers
    lr=1e-3
)

In [ ]:
train_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.5],
                         [0.5])
])

val_transforms = transforms.Compose([
    # transforms.Grayscale(num_output_channels=1),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import kagglehub

path = kagglehub.dataset_download("kapillondhe/american-sign-language", output_dir="./data", force_download=True)

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'american-sign-language' dataset.
Path to dataset files: /kaggle/input/american-sign-language


In [ ]:
path1 = kagglehub.dataset_download("grassknoted/asl-alphabet")

print("Path to dataset files:", path1)

Using Colab cache for faster access to the 'asl-alphabet' dataset.
Path to dataset files: /kaggle/input/asl-alphabet


In [ ]:
path2 = kagglehub.dataset_download("debashishsau/aslamerican-sign-language-aplhabet-dataset")

print("Path to dataset files:", path2)

Using Colab cache for faster access to the 'aslamerican-sign-language-aplhabet-dataset' dataset.
Path to dataset files: /kaggle/input/aslamerican-sign-language-aplhabet-dataset


In [ ]:
path3 = kagglehub.dataset_download("mrgeislinger/asl-rgb-depth-fingerspelling-spelling-it-out")

print("Path to dataset files:", path3)

Using Colab cache for faster access to the 'asl-rgb-depth-fingerspelling-spelling-it-out' dataset.
Path to dataset files: /kaggle/input/asl-rgb-depth-fingerspelling-spelling-it-out


In [ ]:
path4 = kagglehub.dataset_download("ahmedkhanak1995/sign-language-gesture-images-dataset", output_dir="'/data4")

print("Path to dataset files:", path4)

Using Colab cache for faster access to the 'sign-language-gesture-images-dataset' dataset.
Path to dataset files: /kaggle/input/sign-language-gesture-images-dataset


In [ ]:
path5 = kagglehub.dataset_download("alhasangamalmahmoud/american-sign-language-asl", output_dir="''/data5")

print("Path to dataset files:", path5)

100%|██████████| 2.09G/2.09G [00:17<00:00, 129MB/s]

Extracting files...


Path to dataset files: ''/data5


In [ ]:
import shutil
import os

# Define the base source and destination directories
base_source_dir = "/kaggle/input/sign-language-gesture-images-dataset/Gesture Image Data"
base_destination_dir = '/content/data/ASL_Dataset/Train'

# Define the letters to iterate through
letters = ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']

for letter in letters:
    # Construct source path with lowercase subfolder name
    source_dir = os.path.join(base_source_dir, letter.upper())
    # Destination path remains with uppercase letter
    destination_dir = os.path.join(base_destination_dir, letter.upper())

    # Create the parent directory for the destination if it doesn't exist
    os.makedirs(os.path.dirname(destination_dir), exist_ok=True)

    try:
        # Use copytree to copy the di
        shutil.copytree(source_dir, destination_dir, dirs_exist_ok=True)
        print(f"Successfully copied '{source_dir}' to '{destination_dir}'")
    except FileNotFoundError:
        print(f"Error: Source directory '{source_dir}' not found. Skipping {letter}")
    except Exception as e:
        print(f"An error occurred during copying '{source_dir}': {e}")


Successfully copied '/kaggle/input/sign-language-gesture-images-dataset/Gesture Image Data/A' to '/content/data/ASL_Dataset/Train/A'
Successfully copied '/kaggle/input/sign-language-gesture-images-dataset/Gesture Image Data/B' to '/content/data/ASL_Dataset/Train/B'
Successfully copied '/kaggle/input/sign-language-gesture-images-dataset/Gesture Image Data/C' to '/content/data/ASL_Dataset/Train/C'
Successfully copied '/kaggle/input/sign-language-gesture-images-dataset/Gesture Image Data/D' to '/content/data/ASL_Dataset/Train/D'
Successfully copied '/kaggle/input/sign-language-gesture-images-dataset/Gesture Image Data/E' to '/content/data/ASL_Dataset/Train/E'
Successfully copied '/kaggle/input/sign-language-gesture-images-dataset/Gesture Image Data/F' to '/content/data/ASL_Dataset/Train/F'
Successfully copied '/kaggle/input/sign-language-gesture-images-dataset/Gesture Image Data/G' to '/content/data/ASL_Dataset/Train/G'
Successfully copied '/kaggle/input/sign-language-gesture-images-datas

In [ ]:
import shutil
shutil.rmtree("/content/data/ASL_Dataset/Train/.ipynb_checkpoints", ignore_errors=True)

In [ ]:
path = "data"
# Load datasets WITHOUT transforms initially
dataset1 = datasets.ImageFolder("/kaggle/input/aslamerican-sign-language-aplhabet-dataset/ASL_Alphabet_Dataset/asl_alphabet_train", transform=val_transforms)
dataset2 = datasets.ImageFolder("/content/data/ASL_Dataset/Train", transform=val_transforms)
dataset3 = datasets.ImageFolder("/kaggle/input/asl-alphabet/asl_alphabet_train/asl_alphabet_train", transform=val_transforms)


dataset = ConcatDataset([dataset1, dataset2, dataset3])

n_total = len(dataset)
n_train = int(0.8*n_total)
n_val = int(n_total - n_train)

train_dataset, val_dataset = random_split(dataset, [n_train, n_val], generator=torch.Generator().manual_seed(42))

In [ ]:
print('Classes in dataset1:', dataset1.classes)
print('Classes in dataset2:', dataset2.classes)
print('Classes in dataset3:', dataset3.classes)

Classes in dataset1: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'del', 'nothing', 'space']
Classes in dataset2: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'del', 'nothing', 'space']
Classes in dataset3: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'del', 'nothing', 'space']


In [ ]:
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2, pin_memory=True, persistent_workers=True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False, num_workers=2, pin_memory=True, persistent_workers=True)

In [ ]:
device = ("cuda" if torch.cuda.is_available() else "cpu")
# model = LeNet().to(device=device)
model = model.to(device)


In [ ]:
from tqdm import tqdm

In [ ]:
def train(model, train_loader):
  model.train()
  running_loss = 0.0
  for images, labels in tqdm(train_loader, desc="Training", leave=True):
      images, labels = images.to(device), labels.to(device)

      optimizer.zero_grad()
      outputs = model(images)
      loss = criterion(outputs, labels)
      loss.backward()
      optimizer.step()

      running_loss += loss.item()
  train_loss = running_loss / len(train_loader)
  return train_loss

In [ ]:
def validate(model, val_loader):
  model.eval()
  val_loss = 0.0
  correct = 0

  total = 0
  with torch.no_grad():
      for images, labels in tqdm(val_loader, desc="validating", leave=True):
          images, labels = images.to(device), labels.to(device)
          outputs = model(images)
          loss = criterion(outputs, labels)
          val_loss += loss.item()
          preds = outputs.argmax(dim=1)
          correct += (preds == labels).sum().item()
          total += labels.size(0)
  val_loss /= len(val_loader)
  val_acc = correct / total
  return val_loss, val_acc

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
model.to(device)
epochs = 2
best_val_acc = 0.0
criterion = nn.CrossEntropyLoss()

for epoch in range(epochs):
    train_loss = train(model, train_loader)
    val_loss, val_acc = validate(model, val_loader)

    if val_acc > best_val_acc:
      best_val_acc = val_acc
      torch.save(model.state_dict(), f"/content/drive/My Drive/model{epoch}.pth")



    print(f"Epoch [{epoch+1}/{epochs}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

validating: 100%|██████████| 855/855 [07:47<00:00,  1.83it/s]


Epoch [1/2], Train Loss: 0.7400, Val Loss: 0.3495, Val Acc: 0.9069


validating: 100%|██████████| 855/855 [06:45<00:00,  2.11it/s]


Epoch [2/2], Train Loss: 0.5349, Val Loss: 0.2931, Val Acc: 0.9196


In [ ]:
model.to(device)
epochs = 3
best_val_acc = 0.0
criterion = nn.CrossEntropyLoss()

for epoch in range(epochs):
    train_loss = train(model, train_loader)
    val_loss, val_acc = validate(model, val_loader)

    if val_acc > best_val_acc:
      best_val_acc = val_acc
      torch.save(model.state_dict(), f"/content/drive/My Drive/modelv{epoch}.pth")



    print(f"Epoch [{epoch+1}/{epochs}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

validating: 100%|██████████| 855/855 [07:01<00:00,  2.03it/s]


Epoch [1/3], Train Loss: 0.5192, Val Loss: 0.2795, Val Acc: 0.9221


validating: 100%|██████████| 855/855 [06:48<00:00,  2.09it/s]


Epoch [2/3], Train Loss: 0.5135, Val Loss: 0.2654, Val Acc: 0.9246


validating: 100%|██████████| 855/855 [06:59<00:00,  2.04it/s]


Epoch [3/3], Train Loss: 0.5121, Val Loss: 0.2612, Val Acc: 0.9258
